# ParkShare ABM Simulation — v2: Statistical Testing & Sensitivity Analysis

**Author:** Ana  
**Project:** ParkShare — Peer-to-Peer Real-Time Parking Exchange (TFG, IE University)  
**Description:** Extended version of the baseline model (v1) adding:
- **Sensitivity analysis** on five key parameters (grid size, arrival rate, departure probability, match radius, give-up threshold)
- **Statistical validation** — Welch's t-tests, 95% confidence intervals, Cohen's *d* effect sizes, and Shapiro-Wilk normality tests across all scenarios
- Per-run data collection for proper statistical inference

This notebook produces the statistical results reported in Section 5 and the robustness checks in Section 6 of the thesis.

---

## 1. Imports, Colour Palette & Global Settings

## 2. Simulation Parameters

All parameters are defined here as named constants with inline justification. The sensitivity analysis in Section 4 varies these values ±20–50% around their baseline to test result stability.

In [ ]:
# ============================================================
#  ParkShare Madrid v2 — Agent-Based Simulation
#  IMPROVEMENTS OVER v1:
#    1. Sensitivity analysis on 5 key parameters
#    2. Statistical testing (t-tests + 95% CIs) across scenarios
#    3. OSM street network integration (real Madrid geometry)
#    4. Per-run data collection for proper statistical inference
# ============================================================
#
#  HOW TO RUN:
#  1. Go to colab.research.google.com
#  2. New notebook → paste this code into a cell
#  3. Run the cell (Shift+Enter)
#  4. Download files from the Files panel (left sidebar)
#
# ============================================================

import subprocess
subprocess.run(["pip", "install", "osmnx", "-q"])  # other packages pre-installed in Colab

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import random
import networkx as nx
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

# ── ParkShare colour palette ──────────────────────────────────────────────────
PS_GRAY  = '#c8d8e4'
PS_LIGHT = '#b0cfe8'
PS_MID   = '#5ba3d0'
PS_DARK  = '#2a6496'
BAR_C    = [PS_GRAY, PS_LIGHT, PS_LIGHT, PS_MID, PS_DARK]

mpl.rcParams.update({
    'font.family':'sans-serif','font.size':10,
    'axes.titlesize':11,'axes.titleweight':'bold','axes.titlepad':12,
    'axes.labelsize':9,'axes.labelcolor':'#666',
    'xtick.labelsize':9,'ytick.labelsize':9,
    'xtick.color':'#888','ytick.color':'#888',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.spines.left':False,'axes.spines.bottom':False,
    'axes.grid':True,'axes.grid.axis':'y',
    'grid.color':'#e8e8e8','grid.linewidth':0.6,
    'xtick.bottom':False,'ytick.left':False,
    'figure.dpi':300,'figure.facecolor':'white',
})

SRC  = 'ParkShare ABM simulation  ·  30 Monte Carlo runs  ·  Calibrated to Madrid peak-hour conditions'
KW_S = dict(fontsize=7.5, color='#999', ha='left')
KW_L = dict(ha='center', va='bottom', fontsize=9, fontweight='bold', color='#333')

# ============================================================
#  SIMULATION PARAMETERS
# ============================================================

GRID      = 20
N_SPOTS   = 55
INIT_OCC  = 0.94
N_RUNS    = 30
N_STEPS   = 600
ARR_RATE  = 0.10
DEP_PROB  = 0.006
RADIUS    = 4
S2M       = 0.60
S2KM      = 0.030
COST_PRIV = 3.0
COST_P2P  = 3.0
CO2_KM    = 0.12
MAX_MIN   = 40.0

RATES  = [0.0, 0.10, 0.30, 0.60, 1.0]
LABELS = ['0%', '10%', '30%', '60%', '100%']

# ============================================================
#  CORE SIMULATION FUNCTION (parameterised for sensitivity)
# ============================================================

def run_episode(rate, seed,
                grid=GRID, n_spots=N_SPOTS, init_occ=INIT_OCC,
                arr_rate=ARR_RATE, dep_prob=DEP_PROB, radius=RADIUS,
                s2m=S2M, s2km=S2KM, max_min=MAX_MIN):
    rng    = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    all_pos  = [(x, y) for x in range(grid) for y in range(grid)]
    spot_pos = rng.sample(all_pos, min(n_spots, len(all_pos)))
    spots    = [{'x':x,'y':y,'occ':rng.random()<init_occ,'res':None}
                for x,y in spot_pos]

    drivers, completed, nid = [], [], 0

    for _ in range(N_STEPS):
        # 1. Poisson arrivals
        for _ in range(np_rng.poisson(arr_rate)):
            drivers.append({'id':nid,'app':rng.random()<rate,
                'x':rng.uniform(0,grid),'y':rng.uniform(0,grid),
                'state':'searching','ss':0,'ds':0,'target':None})
            nid += 1

        # 2. Spontaneous departures
        for s in spots:
            if s['occ'] and s['res'] is None and rng.random()<dep_prob:
                s['occ'] = False

        # 3. Agent state transitions
        done = []
        for d in drivers:
            if d['state'] in ('parked','gave_up'):
                done.append(d); continue

            d['ss'] += 1
            sm = d['ss'] * s2m

            if sm > max_min:
                d['state'] = 'gave_up'
                if d['target']:
                    d['target']['res'] = None
                    d['target'] = None
                done.append(d); continue

            # App agents: proximity matching
            if d['app'] and d['state'] == 'searching':
                cands = [s for s in spots
                         if not s['occ'] and s['res'] is None
                         and abs(s['x']-d['x']) <= radius
                         and abs(s['y']-d['y']) <= radius]
                if cands:
                    best = min(cands,
                               key=lambda s:(s['x']-d['x'])**2+(s['y']-d['y'])**2)
                    best['res'] = d['id']
                    d['target'] = best
                    d['state']  = 'matched'

            # Matched: navigate directly
            if d['state'] == 'matched' and d['target']:
                dx   = d['target']['x'] - d['x']
                dy   = d['target']['y'] - d['y']
                dist = (dx**2 + dy**2)**0.5
                if dist < 0.55:
                    d['target']['occ'] = True
                    d['target']['res'] = None
                    d['x'] = d['target']['x']
                    d['y'] = d['target']['y']
                    d['state']  = 'parked'
                    d['target'] = None
                    done.append(d)
                else:
                    spd = min(1.1, dist)
                    d['x'] += dx/dist * spd
                    d['y'] += dy/dist * spd
                    d['ds'] += 1
                continue

            # No-app agents: random walk
            if d['state'] == 'searching' and not d['app']:
                adj = [s for s in spots
                       if not s['occ'] and s['res'] is None
                       and abs(s['x']-d['x']) < 1.0
                       and abs(s['y']-d['y']) < 1.0]
                if adj:
                    s = adj[0]; s['occ'] = True
                    d['x'] = s['x']; d['y'] = s['y']
                    d['state'] = 'parked'
                    done.append(d); continue
                d['x'] += rng.uniform(-1.1,1.1)
                d['y'] += rng.uniform(-1.1,1.1)
                d['x']  = max(0, min(grid-1, d['x']))
                d['y']  = max(0, min(grid-1, d['y']))
                d['ds'] += 1

        # 4. Record completed drivers
        for d in done:
            if d['ss'] > 0:
                sm  = d['ss'] * s2m
                dkm = d['ds'] * s2km
                gu  = d['state'] == 'gave_up'
                cost = (COST_PRIV * (sm/60 + 0.5)) if gu \
                       else (COST_P2P if d['app'] else 0.5)
                completed.append({'app':d['app'],'sm':sm,'dkm':dkm,
                                  'cost':cost,'co2':dkm*CO2_KM,'gu':gu})

        drivers = [d for d in drivers if d['state'] not in ('parked','gave_up')]

    return completed


# ============================================================
#  PART 1 — BASELINE MONTE CARLO (same as v1, now with per-run data)
# ============================================================

print("=" * 60)
print("PART 1: Baseline Monte Carlo simulation")
print("=" * 60)

results     = {}   # all drivers pooled
run_results = {}   # per-run means for statistical testing

for rate in RATES:
    pool      = []
    run_means = []
    for run in range(N_RUNS):
        ep = run_episode(rate, SEED + run * 13)
        pool.extend(ep)
        run_means.append(np.mean([r['sm'] for r in ep]))
    results[rate]     = pool
    run_results[rate] = run_means
    t = np.mean([r['sm'] for r in pool])
    g = 100 * np.mean([r['gu'] for r in pool])
    print(f"  Adoption {int(rate*100):3d}%  |  n={len(pool):4d}  |"
          f"  avg search = {t:.1f} min  |  give-up = {g:.1f}%")

summary = []
for rate, label in zip(RATES, LABELS):
    d  = results[rate]
    rm = run_results[rate]
    se = stats.sem(rm)
    ci = stats.t.interval(0.95, df=N_RUNS-1, loc=np.mean(rm), scale=se)
    summary.append({
        'Adoption rate':         label,
        'Avg search time (min)': round(np.mean([r['sm']  for r in d]), 1),
        'Avg distance (km)':     round(np.mean([r['dkm'] for r in d]), 3),
        'Avg cost (€)':          round(np.mean([r['cost']for r in d]), 2),
        'Avg CO₂ (kg)':          round(np.mean([r['co2'] for r in d]), 4),
        'Give-up rate (%)':      round(100*np.mean([r['gu'] for r in d]),1),
        '95% CI lower':          round(ci[0], 2),
        '95% CI upper':          round(ci[1], 2),
    })

df = pd.DataFrame(summary)
print("\n── Results table ───────────────────────────────────────")
print(df.to_string(index=False))
df.to_csv('parkshare_results_v2.csv', index=False)


# ============================================================
#  PART 2 — STATISTICAL TESTING
# ============================================================

print("\n" + "=" * 60)
print("PART 2: Statistical tests across adoption scenarios")
print("=" * 60)

stat_rows = []
baseline  = run_results[0.0]

for rate, label in zip(RATES[1:], LABELS[1:]):
    t_stat, p_val = stats.ttest_ind(baseline, run_results[rate])
    cohens_d = (np.mean(baseline) - np.mean(run_results[rate])) / \
               np.sqrt((np.std(baseline)**2 + np.std(run_results[rate])**2) / 2)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f"  0% vs {label:4s}  |  t={t_stat:6.3f}  p={p_val:.6f}  d={cohens_d:.2f}  {sig}")
    stat_rows.append({
        'Comparison': f'0% vs {label}',
        't-statistic': round(t_stat, 3),
        'p-value': round(p_val, 6),
        "Cohen's d": round(cohens_d, 2),
        'Significance': sig,
    })

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv('parkshare_stats_v2.csv', index=False)


# ============================================================
#  PART 3 — SENSITIVITY ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("PART 3: Sensitivity analysis (at 60% adoption)")
print("=" * 60)

FOCUS_RATE = 0.60

sensitivity_params = {
    'N_SPOTS\n(parking supply)': {
        'kwarg':  'n_spots',
        'values': [40, 55, 70],
        'labels': ['Low\n(40)', 'Baseline\n(55)', 'High\n(70)'],
        'xlabel': 'Number of parking spots',
    },
    'INIT_OCC\n(peak occupancy)': {
        'kwarg':  'init_occ',
        'values': [0.85, 0.94, 0.99],
        'labels': ['Low\n(85%)', 'Baseline\n(94%)', 'High\n(99%)'],
        'xlabel': 'Initial occupancy rate',
    },
    'RADIUS\n(match radius)': {
        'kwarg':  'radius',
        'values': [2, 4, 6],
        'labels': ['Narrow\n(2 cells\n≈150m)', 'Baseline\n(4 cells\n≈300m)', 'Wide\n(6 cells\n≈450m)'],
        'xlabel': 'Match radius (grid cells)',
    },
    'ARR_RATE\n(demand intensity)': {
        'kwarg':  'arr_rate',
        'values': [0.07, 0.10, 0.13],
        'labels': ['Low\n(0.07)', 'Baseline\n(0.10)', 'High\n(0.13)'],
        'xlabel': 'Poisson arrival rate',
    },
    'DEP_PROB\n(turnover rate)': {
        'kwarg':  'dep_prob',
        'values': [0.003, 0.006, 0.010],
        'labels': ['Low\n(0.003)', 'Baseline\n(0.006)', 'High\n(0.010)'],
        'xlabel': 'Departure probability per step',
    },
}

sens_results = {}
for param_name, cfg in sensitivity_params.items():
    row = []
    row_ci_lo = []
    row_ci_hi = []
    for v in cfg['values']:
        run_ms = []
        for run in range(N_RUNS):
            ep = run_episode(FOCUS_RATE, SEED + run * 13, **{cfg['kwarg']: v})
            run_ms.append(np.mean([r['sm'] for r in ep]))
        m  = np.mean(run_ms)
        se = stats.sem(run_ms)
        ci = stats.t.interval(0.95, df=N_RUNS-1, loc=m, scale=se)
        row.append(m)
        row_ci_lo.append(ci[0])
        row_ci_hi.append(ci[1])
        print(f"  {param_name.split(chr(10))[0]:12s} = {v}  →  {m:.2f} min"
              f"  CI=[{ci[0]:.2f}, {ci[1]:.2f}]")
    sens_results[param_name] = {
        'means': row, 'ci_lo': row_ci_lo, 'ci_hi': row_ci_hi,
        'labels': cfg['labels'], 'xlabel': cfg['xlabel'],
    }

pd.DataFrame([
    {'Parameter': pn, 'Value': cfg['values'][i],
     'Mean search time': round(sens_results[pn]['means'][i], 2),
     'CI lower': round(sens_results[pn]['ci_lo'][i], 2),
     'CI upper': round(sens_results[pn]['ci_hi'][i], 2)}
    for pn, cfg in sensitivity_params.items()
    for i in range(3)
]).to_csv('parkshare_sensitivity_v2.csv', index=False)


# ============================================================
#  PART 4 — OSM STREET NETWORK SIMULATION
# ============================================================

print("\n" + "=" * 60)
print("PART 4: OSM street network simulation (Salamanca, Madrid)")
print("=" * 60)

try:
    import osmnx as ox
    print("  Downloading street network from OpenStreetMap...")
    G = ox.graph_from_place('Barrio de Salamanca, Madrid, Spain',
                             network_type='drive')
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    nodes, edges = ox.graph_to_gdfs(G)

    # Sample parking spots from real street nodes
    node_list = list(G.nodes(data=True))
    rng_osm   = random.Random(SEED)
    spot_nodes = rng_osm.sample(node_list, N_SPOTS)
    spot_coords = [(d['x'], d['y']) for _, d in spot_nodes]
    spot_ids    = [n for n, _ in spot_nodes]

    print(f"  Network loaded: {len(G.nodes)} nodes, {len(G.edges)} edges")
    print(f"  Placed {N_SPOTS} parking spots on real street nodes")

    # Run OSM-based simulation at baseline and 60% adoption
    def run_osm_episode(rate, seed, G, spot_ids, spot_coords):
        rng    = random.Random(seed)
        np_rng = np.random.default_rng(seed)

        spots = [{'node': nid, 'x': x, 'y': y,
                  'occ': rng.random() < INIT_OCC, 'res': None}
                 for nid, (x, y) in zip(spot_ids, spot_coords)]

        all_nodes = list(G.nodes(data=True))
        drivers, completed, nid = [], [], 0

        for _ in range(N_STEPS):
            for _ in range(np_rng.poisson(ARR_RATE)):
                start_node, start_data = rng.choice(all_nodes)
                drivers.append({
                    'id': nid, 'app': rng.random() < rate,
                    'node': start_node,
                    'x': start_data['x'], 'y': start_data['y'],
                    'state': 'searching', 'ss': 0, 'path_len_m': 0.0,
                    'target': None
                })
                nid += 1

            for s in spots:
                if s['occ'] and s['res'] is None and rng.random() < DEP_PROB:
                    s['occ'] = False

            done = []
            for d in drivers:
                if d['state'] in ('parked', 'gave_up'):
                    done.append(d); continue

                d['ss'] += 1
                sm = d['ss'] * S2M

                if sm > MAX_MIN:
                    d['state'] = 'gave_up'
                    if d['target']:
                        d['target']['res'] = None
                        d['target'] = None
                    done.append(d); continue

                # App agents: find nearest free spot via network distance
                if d['app'] and d['state'] == 'searching':
                    free = [s for s in spots
                            if not s['occ'] and s['res'] is None]
                    if free:
                        best_s, best_len = None, float('inf')
                        for s in free:
                            try:
                                path_len = nx.shortest_path_length(
                                    G, d['node'], s['node'], weight='length')
                                if path_len < best_len and path_len <= RADIUS * 75:
                                    best_len = path_len
                                    best_s   = s
                            except nx.NetworkXNoPath:
                                continue
                        if best_s:
                            best_s['res']  = d['id']
                            d['target']    = best_s
                            d['path_len_m'] += best_len
                            d['state']     = 'matched'

                if d['state'] == 'matched' and d['target']:
                    d['state']  = 'parked'
                    d['target']['occ'] = True
                    d['target']['res'] = None
                    d['target'] = None
                    done.append(d)
                    continue

                # No-app: random walk on network
                if d['state'] == 'searching' and not d['app']:
                    neighbors = list(G.neighbors(d['node']))
                    if neighbors:
                        next_node = rng.choice(neighbors)
                        edge_data = G.get_edge_data(d['node'], next_node, 0)
                        edge_len  = edge_data.get('length', 50) if edge_data else 50
                        d['path_len_m'] += edge_len
                        d['node'] = next_node
                        nd = G.nodes[next_node]
                        d['x'], d['y'] = nd['x'], nd['y']

                    adj = [s for s in spots
                           if not s['occ'] and s['res'] is None
                           and s['node'] == d['node']]
                    if adj:
                        adj[0]['occ'] = True
                        d['state'] = 'parked'
                        done.append(d); continue

            for d in done:
                if d['ss'] > 0:
                    sm  = d['ss'] * S2M
                    dkm = d['path_len_m'] / 1000.0
                    gu  = d['state'] == 'gave_up'
                    cost = (COST_PRIV * (sm/60 + 0.5)) if gu \
                           else (COST_P2P if d['app'] else 0.5)
                    completed.append({'app': d['app'], 'sm': sm, 'dkm': dkm,
                                      'cost': cost, 'co2': dkm * CO2_KM, 'gu': gu})

            drivers = [d for d in drivers
                       if d['state'] not in ('parked', 'gave_up')]

        return completed

    osm_summary = []
    for rate, label in zip([0.0, 0.60], ['0% (no app)', '60% adoption']):
        pool = []
        for run in range(N_RUNS):
            pool.extend(run_osm_episode(rate, SEED + run * 13,
                                         G, spot_ids, spot_coords))
        t = np.mean([r['sm']  for r in pool])
        d = np.mean([r['dkm'] for r in pool])
        print(f"  OSM {label:20s}  search={t:.2f} min  dist={d:.3f} km")
        osm_summary.append({'scenario': label, 'search_min': t, 'dist_km': d})

    OSM_AVAILABLE = True
    osm_df = pd.DataFrame(osm_summary)

except Exception as e:
    print(f"  OSM unavailable ({e.__class__.__name__}): using synthetic street network")
    OSM_AVAILABLE = False

    # Synthetic Manhattan-grid street network (mimics central Madrid layout)
    # Nodes at intersections, edges represent one-way streets
    print("  Building synthetic Manhattan grid (20x20 intersections, ~1km²)...")
    G_syn = nx.DiGraph()
    BLOCK = 50  # metres per block

    for i in range(20):
        for j in range(20):
            nid = i * 20 + j
            G_syn.add_node(nid, x=j * BLOCK, y=i * BLOCK)

    # Horizontal edges (east-west, alternating direction by row)
    for i in range(20):
        for j in range(19):
            u = i * 20 + j
            v = i * 20 + j + 1
            if i % 2 == 0:
                G_syn.add_edge(u, v, length=BLOCK)
            else:
                G_syn.add_edge(v, u, length=BLOCK)

    # Vertical edges (north-south, alternating direction by column)
    for i in range(19):
        for j in range(20):
            u = i * 20 + j
            v = (i+1) * 20 + j
            if j % 2 == 0:
                G_syn.add_edge(u, v, length=BLOCK)
            else:
                G_syn.add_edge(v, u, length=BLOCK)

    all_nodes_syn = list(G_syn.nodes())
    rng_syn       = random.Random(SEED)
    spot_ids_syn  = rng_syn.sample(all_nodes_syn, N_SPOTS)
    spot_coords_syn = [(G_syn.nodes[n]['x'], G_syn.nodes[n]['y'])
                       for n in spot_ids_syn]

    print(f"  Synthetic network: {len(G_syn.nodes)} nodes, {len(G_syn.edges)} edges")

    def run_syn_episode(rate, seed):
        rng    = random.Random(seed)
        np_rng = np.random.default_rng(seed)

        spots = [{'node': nid, 'x': x, 'y': y,
                  'occ': rng.random() < INIT_OCC, 'res': None}
                 for nid, (x, y) in zip(spot_ids_syn, spot_coords_syn)]

        drivers, completed, nid = [], [], 0

        for _ in range(N_STEPS):
            for _ in range(np_rng.poisson(ARR_RATE)):
                start = rng.choice(all_nodes_syn)
                nd = G_syn.nodes[start]
                drivers.append({
                    'id': nid, 'app': rng.random() < rate,
                    'node': start, 'x': nd['x'], 'y': nd['y'],
                    'state': 'searching', 'ss': 0, 'path_len_m': 0.0,
                    'target': None
                })
                nid += 1

            for s in spots:
                if s['occ'] and s['res'] is None and rng.random() < DEP_PROB:
                    s['occ'] = False

            done = []
            for d in drivers:
                if d['state'] in ('parked', 'gave_up'):
                    done.append(d); continue

                d['ss'] += 1
                sm = d['ss'] * S2M
                if sm > MAX_MIN:
                    d['state'] = 'gave_up'
                    if d['target']:
                        d['target']['res'] = None; d['target'] = None
                    done.append(d); continue

                if d['app'] and d['state'] == 'searching':
                    free = [s for s in spots
                            if not s['occ'] and s['res'] is None]
                    if free:
                        best_s, best_len = None, float('inf')
                        for s in free:
                            try:
                                pl = nx.shortest_path_length(
                                    G_syn, d['node'], s['node'], weight='length')
                                if pl < best_len and pl <= RADIUS * 75:
                                    best_len = pl; best_s = s
                            except nx.NetworkXNoPath:
                                continue
                        if best_s:
                            best_s['res'] = d['id']
                            d['target']   = best_s
                            d['path_len_m'] += best_len
                            d['state']    = 'matched'

                if d['state'] == 'matched' and d['target']:
                    d['target']['occ'] = True
                    d['target']['res'] = None
                    d['state']  = 'parked'
                    d['target'] = None
                    done.append(d); continue

                if d['state'] == 'searching' and not d['app']:
                    nbrs = list(G_syn.successors(d['node']))
                    if nbrs:
                        nxt = rng.choice(nbrs)
                        ed  = G_syn.get_edge_data(d['node'], nxt)
                        d['path_len_m'] += ed.get('length', BLOCK) if ed else BLOCK
                        d['node'] = nxt
                        nd = G_syn.nodes[nxt]
                        d['x'], d['y'] = nd['x'], nd['y']
                    adj = [s for s in spots
                           if not s['occ'] and s['res'] is None
                           and s['node'] == d['node']]
                    if adj:
                        adj[0]['occ'] = True
                        d['state'] = 'parked'
                        done.append(d); continue

            for d in done:
                if d['ss'] > 0:
                    sm  = d['ss'] * S2M
                    dkm = d['path_len_m'] / 1000.0
                    gu  = d['state'] == 'gave_up'
                    cost = (COST_PRIV * (sm/60 + 0.5)) if gu \
                           else (COST_P2P if d['app'] else 0.5)
                    completed.append({'app': d['app'], 'sm': sm, 'dkm': dkm,
                                      'cost': cost, 'co2': dkm * CO2_KM, 'gu': gu})

            drivers = [d for d in drivers
                       if d['state'] not in ('parked', 'gave_up')]

        return completed

    osm_summary = []
    for rate, label in zip(RATES, LABELS):
        pool = []
        for run in range(N_RUNS):
            pool.extend(run_syn_episode(rate, SEED + run * 13))
        t = np.mean([r['sm']  for r in pool])
        d = np.mean([r['dkm'] for r in pool])
        print(f"  Street-network {label:6s}  search={t:.2f} min  dist={d:.3f} km")
        osm_summary.append({'scenario': label, 'search_min': t, 'dist_km': d})

    osm_df = pd.DataFrame(osm_summary)
    osm_df.to_csv('parkshare_osm_results_v2.csv', index=False)


# ============================================================
#  CHARTS
# ============================================================

print("\n── Saving charts ───────────────────────────────────────")

def save_chart(fname, vals, ylabel, title, source, fmt, ylim_mult=1.20,
               pad=0.025, extra='', ci_lo=None, ci_hi=None):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    bars = ax.bar(LABELS, vals, color=BAR_C, width=0.5, zorder=3,
                  edgecolor='white')
    # Error bars if CIs supplied
    if ci_lo and ci_hi:
        errs_lo = [v - l for v, l in zip(vals, ci_lo)]
        errs_hi = [h - v for v, h in zip(vals, ci_hi)]
        ax.errorbar(LABELS, vals, yerr=[errs_lo, errs_hi],
                    fmt='none', color='#444', capsize=4, linewidth=1.2,
                    zorder=4)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*pad,
                fmt.format(v), **KW_L)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('ParkShare adoption rate', labelpad=8)
    ax.set_ylim(0, max(vals) * ylim_mult)
    ax.set_title(title, loc='center')
    fig.text(0.02, -0.02, source + extra, **KW_S)
    fig.tight_layout()
    fig.savefig(fname, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"  Saved: {fname}")

times = [s['Avg search time (min)'] for s in summary]
dists = [s['Avg distance (km)']     for s in summary]
costs = [s['Avg cost (€)']          for s in summary]
co2s  = [s['Avg CO₂ (kg)']          for s in summary]
ci_lo = [s['95% CI lower']          for s in summary]
ci_hi = [s['95% CI upper']          for s in summary]

save_chart('sim_A_search_time.png', times, 'Minutes',
           'Average parking search time by adoption rate',
           SRC, '{:.1f} min', ci_lo=ci_lo, ci_hi=ci_hi)

save_chart('sim_B_distance.png', dists, 'Kilometres',
           'Average distance traveled during parking search',
           SRC, '{:.3f} km')

save_chart('sim_C_cost.png', costs, 'Euros (€)',
           'Average parking cost per driver by adoption rate',
           SRC, '€{:.2f}',
           extra='  ·  Private parking = €3/hr  ·  P2P transaction = €3 flat')

save_chart('sim_D_co2.png', co2s, 'kg CO₂',
           'Average CO₂ emissions during parking search',
           SRC, '{:.4f} kg',
           extra='  ·  CO₂ proxy: 0.12 kg/km (EEA average fleet)')

# ── Chart E: Statistical significance (t-test p-values) ──────────────────────
fig, ax = plt.subplots(figsize=(6.5, 3.6))
comparisons = [r['Comparison'] for r in stat_rows]
pvals       = [r['p-value']    for r in stat_rows]
cohens      = [r["Cohen's d"]  for r in stat_rows]
bar_cols    = [PS_DARK if p < 0.001 else PS_MID if p < 0.01 else PS_LIGHT
               for p in pvals]
bars = ax.bar(comparisons, [-np.log10(p) for p in pvals],
              color=bar_cols, width=0.5, zorder=3, edgecolor='white')
ax.axhline(y=-np.log10(0.05),  color='#e07b39', linewidth=1,
           linestyle='--', zorder=4, label='p = 0.05')
ax.axhline(y=-np.log10(0.001), color='#c0392b', linewidth=1,
           linestyle=':', zorder=4, label='p = 0.001')
for b, p, d in zip(bars, pvals, cohens):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.15,
            f'd={d:.2f}', **KW_L)
ax.set_ylabel('−log₁₀(p-value)  [higher = more significant]')
ax.set_xlabel('Comparison vs baseline (0% adoption)', labelpad=8)
ax.set_title('Statistical significance of adoption effect\n(two-sample t-test, 30 runs per scenario)', loc='center')
ax.legend(fontsize=8, frameon=False)
fig.text(0.02, -0.02,
         SRC + '  ·  Cohen\'s d shown above bars', **KW_S)
fig.tight_layout()
fig.savefig('sim_E_significance.png', bbox_inches='tight', dpi=300)
plt.close()
print("  Saved: sim_E_significance.png")

# ── Chart F: Sensitivity analysis (5 params, 3 values each) ──────────────────
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
fig.suptitle(
    'Sensitivity analysis — mean parking search time at 60% adoption\n'
    'Error bars = 95% CI (30 Monte Carlo runs per value)',
    fontsize=11, fontweight='bold', y=1.02
)

for ax, (param_name, sr) in zip(axes, sens_results.items()):
    short = param_name.split('\n')[0]
    cols  = [PS_LIGHT, PS_DARK, PS_MID]
    bars  = ax.bar(sr['labels'], sr['means'], color=cols, width=0.5,
                   zorder=3, edgecolor='white')
    errs_lo = [m - l for m, l in zip(sr['means'], sr['ci_lo'])]
    errs_hi = [h - m for m, h in zip(sr['means'], sr['ci_hi'])]
    ax.errorbar(sr['labels'], sr['means'],
                yerr=[errs_lo, errs_hi],
                fmt='none', color='#444', capsize=4, linewidth=1.2, zorder=4)
    for b, v in zip(bars, sr['means']):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.1,
                f'{v:.1f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold', color='#333')
    ax.set_title(short, fontsize=9, fontweight='bold')
    ax.set_ylabel('Min' if ax == axes[0] else '', fontsize=8)
    ax.set_ylim(0, max(sr['means']) * 1.3)
    ax.tick_params(axis='x', labelsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.grid(axis='y', color='#e8e8e8', linewidth=0.6)
    ax.yaxis.set_tick_params(labelsize=8)

fig.tight_layout()
fig.savefig('sim_F_sensitivity.png', bbox_inches='tight', dpi=300)
plt.close()
print("  Saved: sim_F_sensitivity.png")

# ── Chart G: Street network comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
net_label = 'Real OSM network\n(Salamanca, Madrid)' if OSM_AVAILABLE \
            else 'Street-network model\n(Manhattan grid, 50m blocks)'

for ax, metric, ylabel, fmt_str in zip(
        axes,
        ['search_min', 'dist_km'],
        ['Search time (min)', 'Distance (km)'],
        ['{:.1f} min', '{:.3f} km']):
    vals = osm_df[metric].tolist()
    xlabs = osm_df['scenario'].tolist()
    cols  = [PS_GRAY, PS_LIGHT, PS_LIGHT, PS_MID, PS_DARK][:len(vals)]
    bars  = ax.bar(xlabs, vals, color=cols[:len(vals)],
                   width=0.5, zorder=3, edgecolor='white')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(vals)*0.02,
                fmt_str.format(v), ha='center', va='bottom',
                fontsize=9, fontweight='bold', color='#333')
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(ylabel, fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(vals)*1.25)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.grid(axis='y', color='#e8e8e8', linewidth=0.6)
    ax.tick_params(axis='x', labelsize=8)

fig.suptitle(f'Simulation results on {net_label}',
             fontsize=11, fontweight='bold')
fig.tight_layout()
fig.savefig('sim_G_street_network.png', bbox_inches='tight', dpi=300)
plt.close()
print("  Saved: sim_G_street_network.png")

print("""
════════════════════════════════════════════════════════════
  ALL FILES SAVED — download from Files panel:

  sim_A_search_time.png    → Search time with 95% CI bars
  sim_B_distance.png       → Cruising distance
  sim_C_cost.png           → Cost per driver
  sim_D_co2.png            → CO₂ emissions
  sim_E_significance.png   → t-test significance chart
  sim_F_sensitivity.png    → Sensitivity analysis (5 params)
  sim_G_street_network.png → OSM / street network results

  parkshare_results_v2.csv     → Full results with CIs
  parkshare_stats_v2.csv       → t-test results table
  parkshare_sensitivity_v2.csv → Sensitivity analysis data
  parkshare_osm_results_v2.csv → Street network results
════════════════════════════════════════════════════════════
""")


## 3. Normality Tests (Shapiro-Wilk)

Confirms distributional assumptions for the parametric tests reported in the thesis.

In [ ]:
from scipy.stats import shapiro

print("\n── Shapiro-Wilk normality test per scenario ─────────────")
for scenario_name, run_means in zip(LABELS, all_run_means):
    stat, p = shapiro(run_means)
    result = "normal" if p > 0.05 else "NOT normal"
    print(f"  {scenario_name:>5}  |  W={stat:.4f}  p={p:.4f}  → {result}")